''# 01 — Ingestion Bronze NYC Taxi (2025)

**Plateforme :** Microsoft Fabric · **Source :** NYC TLC Yellow Taxi · **Cible :** OneLake Lakehouse / Files

Ce notebook télécharge directement les fichiers Parquet mensuels du serveur NYC TLC vers la couche **Bronze**. Les octets du fichier source sont conservés, sans nettoyage, réécriture Spark ni conversion Delta.

**Avant de démarrer :** dans Microsoft Fabric, attacher votre Lakehouse à ce notebook, puis le définir comme **Lakehouse par défaut**.

**Mode de fonctionnement :** test sur janvier 2025, puis activer les 12 mois via le paramètre `MONTHS`.

> Ce notebook s'exécute dans Fabric (dans le cloud) et n'a pas besoin que votre PC reste connecté pendant toute l'ingestion, une fois l'exécution lancée. Une exécution programmée nécessite de configurer une planification ou un pipeline Fabric séparément.''

## 1 — Configuration

In [ ]:
from pathlib import Path

YEAR = 2025
MONTHS = list(range(1, 13))  # Test pilote. Pour une année entière : list(range(1, 13))
OVERWRITE = False  # Ne jamais écraser une source brute existante par défaut
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
DATASET = "yellow_tripdata"

LAKEHOUSE_FILES = Path("/lakehouse/default/Files")
BRONZE_DIR = LAKEHOUSE_FILES / "bronze" / "taxi" / str(YEAR)
AUDIT_DIR = LAKEHOUSE_FILES / "bronze" / "_metadata" / "ingestion_logs"

assert all(isinstance(m, int) and 1 <= m <= 12 for m in MONTHS), "Mois invalides"
assert LAKEHOUSE_FILES.is_dir(), (
    "Le Lakehouse par défaut n'est pas monté. Attache-le à ton notebook Fabric, "
    "définis-le par défaut puis redémarre la session."
)
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
print("Destination Bronze :", BRONZE_DIR)
print("Mois sélectionnés :", MONTHS)

## 2 — Téléchargement fiable sans modification des fichiers

Le transfert s'effectue en streaming, par blocs, dans un fichier temporaire `.part`. Une fois le transfert terminé, le notebook vérifie la taille et les signatures Parquet (`PAR1`) avant de publier le fichier définitif. Les mois déjà présents sont ignorés si `OVERWRITE=False`. En cas d’erreur, le fichier temporaire est retiré et une nouvelle tentative est effectuée.

In [ ]:
import os
import time
import uuid
import json
from datetime import datetime, timezone
import requests

RUN_ID = uuid.uuid4().hex
CHUNK_BYTES = 4 * 1024 * 1024
MAX_ATTEMPTS = 3

def parquet_signature_ok(path: Path) -> bool:
    if not path.is_file() or path.stat().st_size < 12:
        return False
    with path.open("rb") as f:
        header = f.read(4)
        f.seek(-4, os.SEEK_END)
        footer = f.read(4)
    return header == b"PAR1" and footer == b"PAR1"

def ingest_month(month: int) -> dict:
    filename = f"{DATASET}_{YEAR}-{month:02d}.parquet"
    url = f"{BASE_URL}/{filename}"
    target = BRONZE_DIR / filename
    event = {
        "run_id": RUN_ID, "dataset": DATASET, "year": YEAR, "month": month,
        "source_url": url, "destination": str(target),
        "ingestion_time_utc": datetime.now(timezone.utc).isoformat(),
        "status": "pending", "size_bytes": None, "attempts": 0, "error": None
    }

    if target.exists() and not OVERWRITE:
        event["status"] = "skipped_existing" if parquet_signature_ok(target) else "invalid_existing"
        event["size_bytes"] = target.stat().st_size
        if event["status"] == "invalid_existing":
            event["error"] = "Fichier existant non valide : intervention manuelle requise"
        return event

    for attempt in range(1, MAX_ATTEMPTS + 1):
        part = target.with_name(target.name + f".{RUN_ID}.part")
        event["attempts"] = attempt
        try:
            with requests.get(url, stream=True, timeout=(20, 120)) as response:
                response.raise_for_status()
                expected = response.headers.get("Content-Length")
                received = 0
                with part.open("wb") as output:
                    for chunk in response.iter_content(chunk_size=CHUNK_BYTES):
                        if chunk:
                            output.write(chunk)
                            received += len(chunk)
            if expected is not None and received != int(expected):
                raise IOError(f"Téléchargement incomplet : {received}/{expected} octets")
            if not parquet_signature_ok(part):
                raise ValueError("Signature Parquet non valide ou fichier incomplet")
            os.replace(part, target)
            event["status"] = "downloaded"
            event["size_bytes"] = received
            event["error"] = None
            return event
        except (requests.RequestException, OSError, ValueError) as exc:
            event["error"] = str(exc)[:500]
            part.unlink(missing_ok=True)
            if attempt < MAX_ATTEMPTS:
                time.sleep(2 ** attempt)
    event["status"] = "failed"
    return event

## 3 — Exécution et journal d’ingestion

Un fichier JSON de suivi est conservé par exécution dans `Files/bronze/_metadata/ingestion_logs/`. Chaque ligne contient le mois, la source, le chemin cible, l’horodatage, le statut et le nombre de tentatives. Il s’agit d’un **audit d’ingestion**, pas d’un lineage automatique au niveau colonne.

In [ ]:
results = []
for month in MONTHS:
    print(f"Ingestion {YEAR}-{month:02d}...")
    result = ingest_month(month)
    results.append(result)
    print(f"  → {result['status']} | {result['size_bytes']} octets")
    if result['error']:
        print("  Détail :", result['error'])

log_path = AUDIT_DIR / f"nyc_taxi_{YEAR}_{RUN_ID}.jsonl"
with log_path.open("w", encoding="utf-8") as log:
    for result in results:
        log.write(json.dumps(result, ensure_ascii=False) + "\n")

print("\nJournal enregistré :", log_path)
print("Bilan :", {s: sum(r['status'] == s for r in results) for s in sorted({r['status'] for r in results})})
if any(r['status'] in {"failed", "invalid_existing"} for r in results):
    print("ATTENTION : certaines ingestions nécessitent une correction.")

In [ ]:
sample = BRONZE_DIR / f"{DATASET}_{YEAR}-01.parquet"
if parquet_signature_ok(sample):
    df = spark.read.parquet(str(sample))
    print("Schéma de janvier :")
    df.printSchema()
    print("Aperçu :")
    display(df.limit(5))
else:
    print("Fichier de janvier absent ou invalide ; lance l'ingestion de janvier avant cette vérification.")